In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline

# 1. Cargar datos desde la Capa Gold (Delta Lake)
# En Databricks, convertimos a Pandas solo al final si el dataset cabe en memoria
df_gold = spark.read.table("workspace.gold.risk_prediction_features").toPandas()

# 2. Selección de Features (Usando las nuevas variables de la capa Gold)
features = ['age', 'male', 'cigsPerDay', 'totChol', 'sysBP', 'glucose', 'pulse_pressure', 'is_highly_educated']
target = 'TenYearCHD'

X = df_gold[features]
y = df_gold[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Iniciar experimento en MLflow
# Esto registrará automáticamente parámetros, métricas y el modelo
mlflow.set_experiment("/Users/daniel.garcia@redfoxit.net/framingham_heart_disease")

with mlflow.start_run(run_name="Logistic_Regression_Gold_Features"):
    
    # Usamos un Pipeline para asegurar que el escalamiento se aplique correctamente en producción
    model_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000))
    ])
    
    # Entrenar
    model_pipeline.fit(X_train, y_train)
    
    # Predecir
    y_pred = model_pipeline.predict(X_test)
    
    # 4. Calcular Métricas
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    
    print(f"Accuracy: {accuracy}")
    
    # Loguear parámetros y métricas manualmente (o usar mlflow.sklearn.autolog())
    mlflow.log_param("features", features)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision_chd", report['1']['precision'])
    
    # 5. Generar y Loguear Artefactos (Gráficos)
    # Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Greens")
    plt.title("Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    
    # 6. Registrar el modelo en Unity Catalog
    # Esto permite que el modelo pase de 'Staging' a 'Production'
    signature = infer_signature(X_test, y_pred)
    
    mlflow.sklearn.log_model(
        sk_model=model_pipeline,
        artifact_path="model",
        signature=signature,
        registered_model_name="workspace.gold.heart_disease_prediction"
    )

print("Entrenamiento completado y modelo registrado en MLflow.")